# RQ4 — calculate the missing UMLS margins

The first margin pass **skipped** two cases. This notebook computes them with the
same PART2 SapBERT (mean-pool, L2) + FAISS index. Nothing is imputed.

**MedMentions encoders.** `output_text` is a CUI code, so it is not a valid query.
Query = `gold_mention`. \(s_1\) = max cosine of forms of the encoder's
`predicted_cui`; \(s_2\) = max cosine of a different CUI. Dummy `confidence=1.0`
is **not** used as \(s_1\).

**BioASQ / SQuAD2.** Query = predicted answer string. There is no assigned CUI,
so \(s_1\) = best CUI in the neighbourhood and \(s_2\) = best other CUI
(answer-text candidate margin). BERT/BioBERT/PubMedBERT still have no QA outputs
and cannot be calculated.

Generative MedMentions / all CADEC margins are left untouched.

In [ ]:
from pathlib import Path
from collections import defaultdict
import json
import gc
import shutil

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer
import faiss

PROJECT_ROOT = Path.home() / "projects" / "Measuring-Semantic-Stability-in-Clinical-LLMs"
assert (PROJECT_ROOT / "config.json").is_file()
assert torch.cuda.is_available(), "CUDA required — refuse CPU for SapBERT"

UNASSIGNED = "UNASSIGNED"
MIN_FORM_LEN = 3
FAISS_TOP_K = 50
FAISS_TOP_K_FALLBACK = 1000
_pool_config_name = f"sapbert_full_len{MIN_FORM_LEN}"
EMB_DIR = Path.home() / "data" / "umls" / "embeddings" / _pool_config_name
_SAP_SRC = (
    Path.home() / "data/hf_cache/hub"
    / "models--cambridgeltl--SapBERT-from-PubMedBERT-fulltext"
    / "snapshots" / "090663c3ae57bf35ffe4d0d468a2a88d03051a4d"
)

ENCODERS = ["BERT-base", "BioBERT", "PubMedBERT"]
MAPPED_PATH = PROJECT_ROOT / "outputs/rq1/intermediate/rq1_all_outputs_mapped.csv"
MM_MARGIN_PATH = PROJECT_ROOT / "outputs/rq1/umls_candidate_margin_medmentions.csv"
QA_PATH = PROJECT_ROOT / "outputs/qa/qa_results_combined.csv"
QA_MARGIN_PATH = PROJECT_ROOT / "outputs/qa/umls_candidate_margin_qa.csv"
OUT_DIR = PROJECT_ROOT / "outputs" / "rq4"
ALL_FIG = PROJECT_ROOT / "all_rq_figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ALL_FIG.mkdir(parents=True, exist_ok=True)

for p in (EMB_DIR / "surface_forms.json", EMB_DIR / "cui_form_pairs.json",
          EMB_DIR / "faiss.index", _SAP_SRC, MAPPED_PATH, MM_MARGIN_PATH, QA_PATH):
    assert Path(p).exists(), p

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Loading FAISS from {EMB_DIR}")
with open(EMB_DIR / "surface_forms.json", "r", encoding="utf-8") as f:
    _unique_forms = json.load(f)
with open(EMB_DIR / "cui_form_pairs.json", "r", encoding="utf-8") as f:
    _form_cui_pairs = [tuple(x) for x in json.load(f)]
_faiss_index = faiss.read_index(str(EMB_DIR / "faiss.index"))
assert len(_unique_forms) == _faiss_index.ntotal

def _norm_cui(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return UNASSIGNED
    s = str(x).strip()
    if s.startswith("UMLS:"):
        s = s[5:]
    if s in {"", "NA", "nan", "None", UNASSIGNED}:
        return UNASSIGNED
    return s

_form_to_cuis = defaultdict(set)
for _c, _f in _form_cui_pairs:
    _form_to_cuis[_f].add(_norm_cui(_c))

print(f"FAISS {_faiss_index.ntotal:,} forms")


In [ ]:
def _mean_pool(last_hidden, attn_mask):
    mask = attn_mask.unsqueeze(-1).expand(last_hidden.size()).float()
    summed = torch.sum(last_hidden * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


def _embed_with_model(model, tokenizer, texts, batch_size=128, max_len=64, desc="sapbert"):
    vecs = []
    model.eval()
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc=desc, unit="batch"):
            batch = [str(t) if pd.notna(t) else "" for t in texts[i:i + batch_size]]
            enc = tokenizer(batch, return_tensors="pt", truncation=True,
                            max_length=max_len, padding=True)
            enc = {k: v.to("cuda", non_blocking=True) for k, v in enc.items()}
            out = model(**enc)
            pooled = _mean_pool(out.last_hidden_state, enc["attention_mask"])
            pooled = torch.nn.functional.normalize(pooled.float(), p=2, dim=1)
            vecs.append(pooled.detach().cpu().numpy().astype(np.float32))
    return np.vstack(vecs) if vecs else np.zeros((0, 768), dtype=np.float32)


def _s1_s2_from_hits(sims, idxs, pred_cui):
    """CUI-level s1 (pred) and s2 (best other) from one FAISS row."""
    s1 = None
    s2 = None
    for sim, idx in zip(sims, idxs):
        if int(idx) < 0:
            continue
        form = _unique_forms[int(idx)]
        if len(form) < MIN_FORM_LEN:
            continue
        cuis = _form_to_cuis.get(form, ())
        sc = float(sim)
        if pred_cui != UNASSIGNED and pred_cui in cuis:
            if s1 is None or sc > s1:
                s1 = sc
        if any(c != pred_cui and c != UNASSIGNED for c in cuis):
            if s2 is None or sc > s2:
                s2 = sc
    return s1, s2


def _top_two_cuis(sims, idxs):
    """Best CUI vs best other CUI (used when there is no assigned predicted_cui)."""
    best = {}
    for sim, idx in zip(sims, idxs):
        if int(idx) < 0:
            continue
        form = _unique_forms[int(idx)]
        if len(form) < MIN_FORM_LEN:
            continue
        sc = float(sim)
        for c in _form_to_cuis.get(form, ()):
            if c == UNASSIGNED:
                continue
            prev = best.get(c)
            if prev is None or sc > prev:
                best[c] = sc
    if not best:
        return UNASSIGNED, 0.0, 0.0
    ranked = sorted(best.items(), key=lambda kv: kv[1], reverse=True)
    c1, s1 = ranked[0]
    s2 = ranked[1][1] if len(ranked) > 1 else 0.0
    return c1, float(s1), float(s2)


print(f"Loading SapBERT from {_SAP_SRC}")
_sap_tok = AutoTokenizer.from_pretrained(str(_SAP_SRC))
_sap_mdl = AutoModel.from_pretrained(str(_SAP_SRC))
_sap_mdl = _sap_mdl.to("cuda").eval().half()
assert next(_sap_mdl.parameters()).device.type == "cuda"
print("SapBERT on", next(_sap_mdl.parameters()).device)


In [ ]:
# --- MedMentions encoders: query gold_mention, not the CUI string -------------
df_mapped = pd.read_csv(MAPPED_PATH, low_memory=False)
df_mapped["predicted_cui"] = df_mapped["predicted_cui"].map(_norm_cui)
df_mapped["is_direct_cui"] = df_mapped["is_direct_cui"].astype(bool)
df_enc = df_mapped[df_mapped["model_name"].isin(ENCODERS)].copy()
assert df_enc["is_direct_cui"].all()
assert df_enc["gold_mention"].notna().all()
assert (df_enc["predicted_cui"] != UNASSIGNED).all()
print(f"encoder mapped rows: {len(df_enc):,}")

mentions = df_enc["gold_mention"].astype(str).tolist()
uniq_mentions = sorted(set(mentions))
print(f"unique gold_mention queries: {len(uniq_mentions):,}")
uniq_vecs = _embed_with_model(_sap_mdl, _sap_tok, uniq_mentions,
                              desc="encoder gold_mention")
D50, I50 = _faiss_index.search(uniq_vecs.astype(np.float32), FAISS_TOP_K)
mention_to_ui = {t: i for i, t in enumerate(uniq_mentions)}

# Exact s1: max SapBERT cosine of the predicted CUI's forms (not "0 if outside top-k")
print("Building CUI → form-index map for exact s1")
_form_index = {f: i for i, f in enumerate(_unique_forms)}
_cui_to_idxs = defaultdict(list)
for _c, _f in _form_cui_pairs:
    _cui_to_idxs[_norm_cui(_c)].append(_form_index[_f])
_emb = np.load(str(EMB_DIR / "embeddings.npy"), mmap_mode="r")

s1_cache = {}
s1_vals = np.empty(len(df_enc), dtype=np.float32)
s2_vals = np.empty(len(df_enc), dtype=np.float32)
preds = df_enc["predicted_cui"].tolist()
n_no_forms = 0
for i, (ment, pred) in enumerate(tqdm(list(zip(mentions, preds)), desc="encoder s1/s2")):
    ui = mention_to_ui[ment]
    _, s2 = _s1_s2_from_hits(D50[ui], I50[ui], pred)
    s2_vals[i] = 0.0 if s2 is None else s2
    key = (ui, pred)
    if key not in s1_cache:
        idxs = _cui_to_idxs.get(pred, [])
        if not idxs:
            s1_cache[key] = 0.0
            n_no_forms += 1
        else:
            vecs = np.asarray(_emb[idxs], dtype=np.float32)
            s1_cache[key] = float(np.max(uniq_vecs[ui] @ vecs.T))
    s1_vals[i] = s1_cache[key]
print(f"unique (mention, CUI) s1 lookups: {len(s1_cache):,}  CUIs with no forms: {n_no_forms}")
print(f"s1 mean={float(np.mean(s1_vals)):.4f}  s2 mean={float(np.mean(s2_vals)):.4f}")
assert float(np.mean(s1_vals)) > 0.05, "s1 collapsed — mention vs predicted-CUI cosine looks empty"

df_enc = df_enc.copy()
df_enc["s1"] = s1_vals
df_enc["s2"] = s2_vals
df_enc["umls_margin"] = df_enc["s1"] - df_enc["s2"]
assert df_enc["umls_margin"].notna().all()
print(df_enc.groupby("model_name")[["s1", "s2", "umls_margin"]].mean().round(4).to_string())
print(df_enc["umls_margin"].describe().round(4).to_string())

orig = (
    df_enc[df_enc["input_type"] == "original"]
    .drop_duplicates(["instance_id", "model_name"], keep="first")
    [["instance_id", "model_name", "umls_margin"]]
    .rename(columns={"umls_margin": "margin_original"})
)
df_margin = (
    df_enc.groupby(["instance_id", "model_name"], as_index=False)
    .agg(
        n_retrieval_rows=("umls_margin", "size"),
        margin_mean=("umls_margin", "mean"),
        margin_min=("umls_margin", "min"),
        s1_mean=("s1", "mean"),
        s2_mean=("s2", "mean"),
    )
    .merge(orig, on=["instance_id", "model_name"], how="left")
)
print("instance-level coverage:\n", df_margin.groupby("model_name").size().to_string())

bak = MM_MARGIN_PATH.with_suffix(".csv.bak_before_encoder_margin")
if not bak.exists():
    shutil.copy2(MM_MARGIN_PATH, bak)
    print("backup", bak)
merged = pd.read_csv(MM_MARGIN_PATH)
enc_mask = merged["model_name"].isin(ENCODERS)
n_enc_na = int(merged.loc[enc_mask, "margin_mean"].isna().sum())
n_gen_na = int(merged.loc[~enc_mask, "margin_mean"].isna().sum())
print(f"NA margin before fill: encoders={n_enc_na}  generatives={n_gen_na}")
assert n_enc_na == int(enc_mask.sum()), "encoder margin already filled — refusing overwrite"
assert n_gen_na <= 5, f"too many generative NA margins: {n_gen_na}"
gen_margin_before = merged.loc[~enc_mask, "margin_mean"].astype(float).to_numpy()

cols = ["n_retrieval_rows", "margin_original", "margin_mean", "margin_min", "s1_mean", "s2_mean"]
key = merged.loc[enc_mask, ["instance_id", "model_name"]].reset_index(drop=True)
filled = key.merge(df_margin, on=["instance_id", "model_name"], how="left")
assert filled["margin_mean"].notna().all(), "encoder merge left NA — instance_id mismatch"
assert len(filled) == int(enc_mask.sum())
for c in cols:
    merged.loc[enc_mask, c] = filled[c].to_numpy()

assert merged.loc[enc_mask, "margin_mean"].notna().all()
assert np.allclose(
    merged.loc[~enc_mask, "margin_mean"].astype(float).to_numpy(),
    gen_margin_before,
    equal_nan=True,
)
merged.to_csv(MM_MARGIN_PATH, index=False)
print(f"Wrote {MM_MARGIN_PATH}  encoder margin defined: {int(enc_mask.sum()):,}")
print(merged.groupby("model_name")["margin_mean"].agg(n="size", defined="count", mean="mean").round(4).to_string())

H = merged["normalised_semantic_entropy_full"].astype(float).clip(lower=0)
zero = merged[(H == 0) & merged["model_name"].isin(ENCODERS)]
print(f"H=0 encoder rows: {len(zero):,}  margin std={zero['margin_mean'].std(ddof=1):.4f}")


In [ ]:
# --- QA: candidate margin of the predicted answer string ---------------------
qa = pd.read_csv(QA_PATH)
qa = qa[qa["m"] >= 3].copy()
qa["pred"] = qa["pred"].fillna("").astype(str)
print(f"QA included rows: {len(qa):,}  unique preds: {qa['pred'].nunique():,}")
print("QA models (encoders were never run):", sorted(qa.model.unique()))

uniq_pred = qa["pred"].unique().tolist()
pred_vecs = _embed_with_model(_sap_mdl, _sap_tok, uniq_pred, desc="qa pred")
Dq, Iq = _faiss_index.search(pred_vecs.astype(np.float32), FAISS_TOP_K)
pred_to_ui = {t: i for i, t in enumerate(uniq_pred)}

rows = []
for _, r in qa.iterrows():
    ui = pred_to_ui[r["pred"]]
    cui, s1, s2 = _top_two_cuis(Dq[ui], Iq[ui])
    rows.append({
        "id": r["id"], "model": r["model"], "dataset": r["dataset"],
        "pred": r["pred"], "correct": r["correct"],
        "m": r["m"], "norm_entropy": r["norm_entropy"],
        "confidence": r["confidence"],
        "assigned_cui": cui, "s1": s1, "s2": s2,
        "margin_mean": s1 - s2,
        "query": "pred_answer",
        "k": FAISS_TOP_K,
    })
qa_margin = pd.DataFrame(rows)
assert qa_margin["margin_mean"].notna().all()
QA_MARGIN_PATH.parent.mkdir(parents=True, exist_ok=True)
qa_margin.to_csv(QA_MARGIN_PATH, index=False)
print(f"Wrote {QA_MARGIN_PATH}  n={len(qa_margin):,}")
print(qa_margin.groupby(["dataset", "model"])["margin_mean"].agg(n="size", mean="mean", std="std").round(4).to_string())

del _sap_mdl, uniq_vecs, pred_vecs, D50, I50, Dq, Iq
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# --- AURC / curves from the calculated margins (same helpers as RQ4) ---------
# Only ADD encoder MM + QA margin series. Do not recompute generative random
# (that would consume RNG and drift stored AURC).
COVERAGE_GRID = np.round(np.arange(1.00, 0.09, -0.05), 2)
N_RANDOM = 20
RNG = np.random.default_rng(42)
SIGNALS = ["entropy", "confidence", "margin", "random", "combined", "combined_3"]

def selective_curve(y, score_order, coverages=COVERAGE_GRID):
    ranked_y = y[score_order]
    n = len(y)
    rows = []
    for cov in coverages:
        k = max(1, int(np.ceil(float(cov) * n)))
        acc = float(np.mean(ranked_y[:k]))
        rows.append({
            "coverage": float(cov), "n_keep": int(k), "n_total": int(n),
            "selective_accuracy": acc, "risk": 1.0 - acc,
        })
    return rows

def aurc_from_curve(coverages, risks):
    c = np.asarray(coverages, dtype=float)
    r = np.asarray(risks, dtype=float)
    order = np.argsort(c)
    trapz = getattr(np, "trapezoid", None) or np.trapz
    return float(trapz(r[order], c[order]))

def percentile_safe(x, higher_is_safer=True):
    rnk = pd.Series(x).rank(method="average", pct=True).to_numpy(dtype=float)
    return rnk if higher_is_safer else (1.0 - rnk)

def run_group(dataset, model, y, h, conf, marg):
    n = len(y)
    h_safe = percentile_safe(h, higher_is_safer=False)
    c_safe = percentile_safe(conf, higher_is_safer=True)
    m_safe = percentile_safe(marg, higher_is_safer=True)
    combined_3 = (h_safe + c_safe + m_safe) / 3.0
    orders = {
        "entropy": np.argsort(h, kind="mergesort"),
        "confidence": np.argsort(-conf, kind="mergesort"),
        "margin": np.argsort(-marg, kind="mergesort"),
        "combined": np.lexsort((-conf, h)),
        "combined_3": np.argsort(-combined_3, kind="mergesort"),
    }
    curve_rows, curves = [], {}
    for sig, order in orders.items():
        curve = selective_curve(y, order)
        for row in curve:
            row.update({"dataset": dataset, "model": model, "signal": sig, "n": n, "small_n": n < 250})
            curve_rows.append(row)
        curves[sig] = curve
    acc_by_cov = {float(c): [] for c in COVERAGE_GRID}
    for _ in range(N_RANDOM):
        perm = RNG.permutation(n)
        ranked_y = y[perm]
        for cov in COVERAGE_GRID:
            k = max(1, int(np.ceil(float(cov) * n)))
            acc_by_cov[float(cov)].append(float(np.mean(ranked_y[:k])))
    rand_curve = []
    for cov in COVERAGE_GRID:
        acc = float(np.mean(acc_by_cov[float(cov)]))
        row = {"dataset": dataset, "model": model, "signal": "random",
               "coverage": float(cov), "n": n, "small_n": n < 250,
               "selective_accuracy": acc, "risk": 1.0 - acc}
        curve_rows.append(row)
        rand_curve.append(row)
    curves["random"] = rand_curve
    aurc_rows = []
    for sig in SIGNALS:
        aurc_rows.append({
            "dataset": dataset, "model": model, "n": n, "signal": sig,
            "AURC": aurc_from_curve([r["coverage"] for r in curves[sig]],
                                    [r["risk"] for r in curves[sig]]),
        })
    return curve_rows, aurc_rows

mm = pd.read_csv(MM_MARGIN_PATH)
enc_curve, enc_aurc = [], []
for model in ENCODERS:
    g = mm[mm["model_name"] == model].dropna(
        subset=["normalised_semantic_entropy_full", "mean_accuracy_full",
                "mapping_confidence", "margin_mean"]
    )
    assert len(g) > 0, model
    cr, ar = run_group(
        "MedMentions", model,
        g["mean_accuracy_full"].to_numpy(dtype=float),
        g["normalised_semantic_entropy_full"].clip(lower=0).to_numpy(dtype=float),
        g["mapping_confidence"].to_numpy(dtype=float),
        g["margin_mean"].to_numpy(dtype=float),
    )
    enc_curve.extend(cr)
    enc_aurc.extend(ar)
enc_curve_df = pd.DataFrame(enc_curve)
enc_aurc_df = pd.DataFrame(enc_aurc)
print("NEW encoder AURC:\n",
      enc_aurc_df.pivot(index=["model", "n"], columns="signal", values="AURC").round(4).to_string())

old_mm_curve = pd.read_csv(OUT_DIR / "rq4_risk_coverage_margin_medmentions.csv")
old_mm_curve = old_mm_curve[~old_mm_curve["model"].isin(ENCODERS)]
mm_curve_df = pd.concat([old_mm_curve, enc_curve_df], ignore_index=True)
mm_curve_df.to_csv(OUT_DIR / "rq4_risk_coverage_margin_medmentions.csv", index=False)
print("MM curve models:", sorted(mm_curve_df.model.unique()))

bench = pd.read_csv(OUT_DIR / "rq4_aurc_margin_benchmark.csv")
# drop any existing MM encoder rows, append calculated ones
bench = bench[~((bench.dataset == "MedMentions") & (bench.model.isin(ENCODERS)))]
add = enc_aurc_df.copy()
add["dataset_label"] = "MedMentions (biomedical literature)"
add["small_n"] = False
singles_map = []
for model, g in add.groupby("model"):
    amap = dict(zip(g.signal, g.AURC))
    singles = [amap["entropy"], amap["confidence"], amap["margin"]]
    best = min(singles)
    best_name = ["entropy", "confidence", "margin"][int(np.argmin(singles))]
    singles_map.append({
        "model": model,
        "best_single_signal": best_name,
        "best_single_AURC": best,
        "combined_3_lt_best_single": amap["combined_3"] < best - 1e-15,
        "combined_3_lt_combined2": amap["combined_3"] < amap["combined"] - 1e-15,
    })
sm = pd.DataFrame(singles_map)
sm["combined_3_lt_both"] = sm["combined_3_lt_best_single"] & sm["combined_3_lt_combined2"]
add = add.merge(sm, on="model")
for col in bench.columns:
    if col not in add.columns:
        add[col] = np.nan
add = add[bench.columns]
out_bench = pd.concat([bench, add], ignore_index=True)
out_bench.to_csv(OUT_DIR / "rq4_aurc_margin_benchmark.csv", index=False)
print("AURC bench MM models:", sorted(out_bench.loc[out_bench.dataset=="MedMentions","model"].unique()))

qa_m = pd.read_csv(QA_MARGIN_PATH)
qa_m["y"] = pd.to_numeric(qa_m["correct"].map({True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}))
qa_m["dataset_label"] = qa_m["dataset"].map({"bioasq": "BioASQ", "squad2": "SQuAD2"})
qa_curve, qa_aurc = [], []
for (ds, model), g in qa_m.groupby(["dataset_label", "model"], sort=False):
    cr, ar = run_group(
        ds, model,
        g["y"].to_numpy(dtype=float),
        pd.to_numeric(g["norm_entropy"]).to_numpy(dtype=float),
        pd.to_numeric(g["confidence"]).to_numpy(dtype=float),
        pd.to_numeric(g["margin_mean"]).to_numpy(dtype=float),
    )
    qa_curve.extend(cr)
    qa_aurc.extend(ar)
qa_curve_df = pd.DataFrame(qa_curve)
qa_aurc_df = pd.DataFrame(qa_aurc)
qa_curve_df.to_csv(OUT_DIR / "rq4_risk_coverage_margin_qa.csv", index=False)
qa_aurc_df.to_csv(OUT_DIR / "rq4_aurc_margin_qa.csv", index=False)
print("QA AURC:\n", qa_aurc_df.pivot(index=["dataset", "model", "n"], columns="signal", values="AURC").round(4).to_string())
print("DONE compute")

